# Build Embeddings

load the per-game retrieval_text, generate sentence embeddings w all-MiniLM-L6-V2, save them, and test retrieval on a few example queries

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

RETRIEVAL_TEXTS_PATH = Path("games_retrieval_texts.csv")
EMBEDDINGS_PATH = Path("game_embeddings.npy")
APPIDS_PATH = Path("game_app_ids.npy")
MODEL_NAME = "all-MiniLM-L6-v2"

assert RETRIEVAL_TEXTS_PATH.exists(), f"Retrieval texts not found: {RETRIEVAL_TEXTS_PATH.resolve()}"
print(RETRIEVAL_TEXTS_PATH.resolve())

c:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\games_retrieval_texts.csv


## Load retrieval texts

In [2]:
games_df = pd.read_csv(RETRIEVAL_TEXTS_PATH)
games_df["retrieval_text"] = games_df["retrieval_text"].fillna("")

print(f"Loaded {len(games_df):,} games with retrieval text.")
games_df[["appid", "name", "retrieval_text"]].head()

Loaded 39,175 games with retrieval text.


,appid,name,retrieval_text
0,10,Counter-Strike,"Name: Counter-Strike\nGenres: Action\nCategories: Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Color Alte..."
1,20,Team Fortress Classic,"Name: Team Fortress Classic\nGenres: Action\nCategories: Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cus..."
2,30,Day of Defeat,"Name: Day of Defeat\nGenres: Action\nCategories: Multi-player, Camera Comfort, Color Alternatives, Custom Volume Con..."
3,40,Deathmatch Classic,"Name: Deathmatch Classic\nGenres: Action\nCategories: Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Color ..."
4,50,Half-Life: Opposing Force,"Name: Half-Life: Opposing Force\nGenres: Action\nCategories: Single-player, Multi-player, Custom Volume Controls, Ad..."


## Load the embedding model

In [3]:
embedder = SentenceTransformer(MODEL_NAME)
print(f"Loaded model: {MODEL_NAME}")

c:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\heloi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:0

Loaded model: all-MiniLM-L6-v2


## Generate embeddings for all games

Run this once. It may take a while.

In [4]:
retrieval_texts = games_df["retrieval_text"].tolist()
game_embeddings = embedder.encode(
    retrieval_texts,
    show_progress_bar=True,
    batch_size=32,
)

game_embeddings.shape

Batches: 100%|██████████| 1225/1225 [28:23<00:00,  1.39s/it]


(39175, 384)

## Save embeddings to disk

In [5]:
np.save(EMBEDDINGS_PATH, game_embeddings)
np.save(APPIDS_PATH, games_df["appid"].to_numpy())

print(f"Saved embeddings to: {EMBEDDINGS_PATH.resolve()}")
print(f"Saved appids to: {APPIDS_PATH.resolve()}")

Saved embeddings to: C:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\game_embeddings.npy
Saved appids to: C:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\game_app_ids.npy


## Reload saved files (sanity check)

In [6]:
saved_embeddings = np.load(EMBEDDINGS_PATH)
saved_appids = np.load(APPIDS_PATH)

saved_embeddings.shape, saved_appids.shape

((39175, 384), (39175,))

## Try your own queries

In [15]:
def search_games(query: str, top_k: int = 10) -> pd.DataFrame:
    query_embedding = embedder.encode([query])
    similarities = cosine_similarity(query_embedding, saved_embeddings).flatten() #compare the query vector w every game vector
    top_indices = similarities.argsort()[-top_k:][::-1]

    results = games_df.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]
    return results[["appid", "name", "similarity", "short_description"]]


search_games("management game where you build and optimize factories or production chains", top_k=5)

,appid,name,similarity,short_description
37985,3679930,Factory Planner,0.660680,Factory Planner is a card-based simulation/strategy game that lets you build a factory using cards. The main goal of the game is to make combinations with different cards so that each player can build different factories and feel the sandbox experience to the fullest.
38519,3873220,Factory Planner: First Sparks,0.607343,"Factory Planner: First Sparks is a card-based simulation and strategy game. In this Prologue version, build your own factory using strategic card combinations and enjoy a unique sandbox experience."
9874,591370,Production Line : Car factory simulation,0.606701,"Production line is the new car factory management/simulation/tycoon game that pushes your organisational and entrepreneurship skills to the limit. Can you build the ultimate optimised, free-flowing car production line whilst beating the competition and still turn a profit?"
29230,2182630,Mob Factory,0.599705,Mob factory is an tower defense and automation game for the lazy adventurer. Smelt down enemy loot into turrets &amp; factories to automate an ever expanding dungeon from the comfort of your tower.
38218,3759820,Farm Manager,0.587672,"Create and manage your own farm. Strategize, expand, plant, grow and harvest the lands and breed prize-worthy animals in this immersive farm management game. Farm Manager is among the most realistic of the strategy genre - so prove yourself and beat your friends and other real life managers."


In [10]:
search_games("cozy game with relationships", top_k=10)

,appid,name,similarity,short_description
20660,1354520,Wild Honesty: A party game for deeper conversations,0.601224,"A party game for deeper conversations where you’ll push the boundaries of friendships to be more emotionally open and vulnerable. Share thoughts, opinions, and feelings in a more meaningful, challenging, and mindful way. Be seen and heard for who you truly are, and build closer relationships."
34418,2897760,Romantic Escapades,0.598823,"'Romantic Escapades' is an ancient Chinese love simulation game. Through the development of the plot, players gradually develop relationships with various beauties in the game, and then gradually develop deeper and deeper intimate relationships."
6892,447190,Pub Encounter,0.585603,'Pub Encounter' is a romance game where you can fall in love with one of several middle-aged men.
7436,467720,Roomie Romance,0.579453,"Will love blossom between you and your housemate, when you start a new job and have to live with her? The choices you make will determine the relationship you have in this love story between two women."
7872,494450,Office lovers,0.577065,It's a romance game where you can experience the thrill of an office romance where you have to keep your relationship secret. What will happen when you're alone with him in the conference room?
32813,2653260,Naughty Tales of Rabbits - A Cuckold RPG,0.568218,"A whimsical RPG adventure where a loving couple must vanquish the four Elemental Demon Lords! There's just one problem- the girlfriend is lusted after by the denizens of this realm, and the boyfriend has a huge cuckold fetish!"
18892,1217390,Some Some Convenience Store,0.567956,A game about starting a relationship while working part-time at a convenience store
7873,494460,Dangerous Relationship,0.561827,'Dangerous Relationship' is a game where you can enjoy the dangerous world of dating a celebrity. Get seduced while you're on the job and try to keep your relationship a secret!
38565,3894390,boyfriend or cake??,0.556182,alternatively: caring romantic partner or delicious confectionery dessert?
30008,2274430,Under Maintenance,0.555903,"An everyday dramedy otome game that asks the tough question: When a game about love goes down for maintenance, can the game of love begin?"


In [13]:
search_games("competitive tactical shooter", top_k=5)


,appid,name,similarity,short_description
18704,1204040,Shoot on Sight,0.654069,"The System built a world of injustice and oppression. Pressed up against the wall, it now fights for its survival. Uphold the law as an elite police officer or join the Stagers to liberate the world, defining the future of society in this multiplayer isometric tactical shooter!"
33593,2769580,Tactical Warfare: Siege Survival,0.630152,"Immerse yourself in Tactical Warfare, a captivating mini-game of the base game where you defend your base against relentless enemy waves. Strategize, fortify, and survive to emerge victorious. Wishlist now for exclusive access to the playtest and full game on November 29, 2024!."
27157,1945060,Run N' Gun,0.610701,Skate and shoot your way through waves of enemies increasing your firepower along the way to become the champion. Run N' Gun is a First-person arcade shooter with a focus on movement. Become the Protagonist Bot and use your skills to survive waves of Antagonist Bots to win it all.
15804,1000540,Tactical Control,0.610180,"Tactical Control: the 7-minute wargame. Simplified strategy with minimalist graphics and quick, intense games. Play short matches against random opponents. Fight tense battles of stealth, anticipation and surprise. Win with strategy, deception and cleverness, not with reflexes and clicking speed."
24475,1678150,World of Shooting,0.609476,"A first-person shooting sports sim from the developers of World of Guns. Collect hundreds of realistic firearms and accessories, beat practical shooting and tactical training campaigns, set global high scores, and build your own ranges!"


## Inspect one retrieved game's full text

In [9]:
pd.set_option("display.max_colwidth", None)
games_df.loc[top_indices[0], ["name", "retrieval_text"]]

name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  Farm Together 2
retrieval_text    Name: Farm Toget